In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,confusion_matrix)
from src.feature_engineering import build_pipeline



In [2]:
df=pd.read_csv("https://raw.githubusercontent.com/awais-DS/Data/refs/heads/main/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [3]:
df=df[["gender","tenure","Contract","Dependents","MonthlyCharges","Churn"]]

In [4]:
x=df.drop(columns=["Churn"])
y=df["Churn"].map({"No":0,"Yes":1})

In [5]:
x.head()

,gender,tenure,Contract,Dependents,MonthlyCharges
0,Female,1,Month-to-month,No,29.85
1,Male,34,One year,No,56.95
2,Male,2,Month-to-month,No,53.85
3,Male,45,One year,No,42.30
4,Female,2,Month-to-month,No,70.70


In [6]:
y.head()

,Churn
0,0
1,0
2,1
3,0
4,1


In [7]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,stratify=y,random_state=0)

In [8]:
print(x_train.shape,y_train.shape)

(5634, 5) (5634,)


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression

def build_pipeline():
    encoders = ColumnTransformer(
        transformers=[
            ("gender", OneHotEncoder(drop="first", handle_unknown="ignore"), ["gender"]),
            ("dependents", OneHotEncoder(drop="first", handle_unknown="ignore"), ["Dependents"]),
            ("contract", OrdinalEncoder(
                categories=[["Month-to-month", "One year", "Two year"]]), ["Contract"]),
        ],
        remainder="passthrough",
    )
    return Pipeline([
        ("encoders", encoders),
        ("model", LogisticRegression(class_weight="balanced",
                                     max_iter=1000, random_state=0)),
    ])

In [12]:
pipeline=build_pipeline()
pipeline.fit(x_train,y_train)

/usr/local/lib/python3.13/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('encoders',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('gender',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['gender']),
                                                 ('dependents',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['Dependents']),
                                                 ('contract',
                                                  OrdinalEncoder(categories=[['Month-to-month',
                                                                              'One '
                                                                              'year',
                                                                              'Two '
                                                                              'year']]),
                                                  ['Contract'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=0))])

In [14]:
pred = pipeline.predict(x_test)
print(f"Accuracy : {accuracy_score(y_test, pred):.4f}")
print(f"Precision: {precision_score(y_test, pred):.4f}")
print(f"Recall   : {recall_score(y_test, pred):.4f}")
print(f"F1 Score : {f1_score(y_test, pred):.4f}")
print(confusion_matrix(y_test, pred))

Accuracy : 0.7331
Precision: 0.4983
Recall   : 0.8075
F1 Score : 0.6163
[[731 304]
 [ 72 302]]


In [15]:
print(len(df))                  # rows before
df = df.drop_duplicates()   # or whatever cleaning you did in the notebook
print(len(df))                  # rows after: test set should now be 1102

7043
6884


In [16]:
print(len(x_train) + len(x_test))   # total rows the experiment used
print(len(x_test))                  # should print 1102

7043
1409
